In [1]:
import os, random, numpy as np, tensorflow as tf, pandas as pd

os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)

In [2]:
base_dir = "dataset/chest_xray/chest_xray"   # change if needed
trainval_dir = os.path.join(base_dir, "train")  # we'll also include val manually below
val_dir_kaggle = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen_eff = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    validation_split=0.2,
    rotation_range=5,
    width_shift_range=0.02,
    height_shift_range=0.02,
    zoom_range=0.05,
    brightness_range=(0.9, 1.1),
    horizontal_flip=False
)

val_datagen_eff = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    validation_split=0.2
)

test_datagen_eff = ImageDataGenerator(
    preprocessing_function=eff_preprocess
)

train_gen_eff = train_datagen_eff.flow_from_directory(
    trainval_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True
)

val_gen_eff = val_datagen_eff.flow_from_directory(
    trainval_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

test_gen_eff = test_datagen_eff.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)


Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


## EfficientNEtB0(stage 1 = frozen backbone)

In [4]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

def build_efficientnetb0(dropout=0.3):
    base = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False  # Stage 1

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auroc"),
            tf.keras.metrics.AUC(name="auprc", curve="PR"),
        ],
    )
    return model

eff = build_efficientnetb0(dropout=0.3)
eff.summary()


2025-12-31 12:09:04.856294: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-12-31 12:09:04.856337: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-12-31 12:09:04.856359: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-12-31 12:09:04.856377: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-31 12:09:04.856395: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [5]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_eff = [
    ModelCheckpoint("best_effb0.keras", monitor="val_auprc", mode="max",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_auprc", mode="max", patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_auprc", mode="max", factor=0.5,
                      patience=2, min_lr=1e-6, verbose=1),
]


## STage 1 Head only

In [8]:
hist1_eff = eff.fit(
    train_gen_eff,
    validation_data=val_gen_eff,
    epochs=10,
    callbacks=callbacks_eff
)


Epoch 1/10


2025-12-31 12:11:09.920537: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step - accuracy: 0.7373 - auprc: 0.8996 - auroc: 0.7616 - loss: 0.4926
Epoch 1: val_auprc improved from None to 0.99015, saving model to best_effb0.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 65s 401ms/step - accuracy: 0.8351 - auprc: 0.9621 - auroc: 0.8961 - loss: 0.3626 - val_accuracy: 0.9051 - val_auprc: 0.9902 - val_auroc: 0.9712 - val_loss: 0.2388 - learning_rate: 0.0010
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.9128 - auprc: 0.9878 - auroc: 0.9646 - loss: 0.2320
Epoch 2: val_auprc improved from 0.99015 to 0.99305, saving model to best_effb0.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 49s 372ms/step - accuracy: 0.9152 - auprc: 0.9896 - auroc: 0.9698 - loss: 0.2149 - val_accuracy: 0.9233 - val_auprc: 0.9931 - val_auroc: 0.9795 - val_loss: 0.1919 - learning_rate: 0.0010
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9187 - auprc: 0.9906 - auroc: 0.9738 - loss: 0.1924
Epoch 3: val_auprc improved from 0.99305 to 0.9946

## Unfreeze last ~30 layers with low leraning rate

In [9]:
eff.load_weights("best_effb0.keras")

base = eff.layers[1]  # EfficientNetB0 backbone
base.trainable = True

# Fine-tune only the last block-ish layers
for layer in base.layers[:-30]:
    layer.trainable = False

eff.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auroc"),
        tf.keras.metrics.AUC(name="auprc", curve="PR"),
    ],
)

hist2_eff = eff.fit(
    train_gen_eff,
    validation_data=val_gen_eff,
    epochs=10,
    callbacks=callbacks_eff
)


Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.8943 - auprc: 0.9910 - auroc: 0.9750 - loss: 0.2712
Epoch 1: val_auprc did not improve from 0.99693
131/131 ━━━━━━━━━━━━━━━━━━━━ 64s 397ms/step - accuracy: 0.9046 - auprc: 0.9907 - auroc: 0.9742 - loss: 0.2540 - val_accuracy: 0.9243 - val_auprc: 0.9959 - val_auroc: 0.9880 - val_loss: 0.1855 - learning_rate: 1.0000e-05
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9313 - auprc: 0.9931 - auroc: 0.9809 - loss: 0.1975
Epoch 2: val_auprc did not improve from 0.99693

Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
131/131 ━━━━━━━━━━━━━━━━━━━━ 45s 334ms/step - accuracy: 0.9310 - auprc: 0.9926 - auroc: 0.9804 - loss: 0.1897 - val_accuracy: 0.9358 - val_auprc: 0.9959 - val_auroc: 0.9881 - val_loss: 0.1715 - learning_rate: 1.0000e-05
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step - accuracy: 0.9419 - auprc: 0.9938 - auroc: 0.9834 - loss: 0.1660
Epoch 3: val_auprc did not 

## Threshold validation

In [14]:
from utils import eval_thresholds


In [15]:
thresholds = np.linspace(0.2, 0.9, 71)
val_table_eff = eval_thresholds(eff, val_gen_eff, thresholds)
val_table_eff


,Threshold,Accuracy,Macro F1,Weighted F1,Specificity,Sensitivity,FN,FP
0,0.20,0.945,0.928,0.945,0.877,0.969,24,33
1,0.21,0.946,0.929,0.946,0.881,0.969,24,32
2,0.22,0.947,0.930,0.947,0.884,0.969,24,31
3,0.23,0.947,0.931,0.947,0.888,0.968,25,30
4,0.24,0.946,0.930,0.946,0.896,0.964,28,28
...,...,...,...,...,...,...,...,...
66,0.86,0.818,0.799,0.829,1.000,0.755,190,0
67,0.87,0.805,0.787,0.817,1.000,0.738,203,0
68,0.88,0.794,0.776,0.807,1.000,0.723,215,0
69,0.89,0.779,0.763,0.793,1.000,0.703,230,0


In [16]:
TARGET_SENS = 0.96
eligible_scr = val_table_eff[val_table_eff["Sensitivity"] >= TARGET_SENS]
best_scr_eff = eligible_scr.sort_values("Specificity", ascending=False).iloc[0]
best_scr_eff


Threshold       0.250
Accuracy        0.946
Macro F1        0.930
Weighted F1     0.946
Specificity     0.899
Sensitivity     0.963
FN             29.000
FP             27.000
Name: 5, dtype: float64

In [17]:
TARGET_SPEC = 0.99
eligible_diag = val_table_eff[val_table_eff["Specificity"] >= TARGET_SPEC]

if len(eligible_diag) == 0:
    # fallback: take max specificity row and within that max sensitivity
    max_spec = val_table_eff["Specificity"].max()
    eligible_diag = val_table_eff[val_table_eff["Specificity"] == max_spec]

best_diag_eff = eligible_diag.sort_values("Sensitivity", ascending=False).iloc[0]
best_diag_eff


Threshold        0.690
Accuracy         0.899
Macro F1         0.881
Weighted F1      0.904
Specificity      0.993
Sensitivity      0.867
FN             103.000
FP               2.000
Name: 49, dtype: float64

In [19]:
from utils import evaluate


In [20]:
best_t_scr = float(best_scr_eff["Threshold"])
best_t_diag = float(best_diag_eff["Threshold"])

eff_screen_res = evaluate(eff, test_gen_eff, name="EfficientNetB0", threshold=best_t_scr)
eff_screen_res["Mode"] = "Screening"
eff_screen_res["Params"] = eff.count_params()

eff_diag_res = evaluate(eff, test_gen_eff, name="EfficientNetB0", threshold=best_t_diag)
eff_diag_res["Mode"] = "Diagnostic"
eff_diag_res["Params"] = eff.count_params()

eff_screen_res, eff_diag_res



=== EfficientNetB0 @ threshold 0.250 ===
[[118 116]
 [  6 384]]
              precision    recall  f1-score   support

      NORMAL       0.95      0.50      0.66       234
   PNEUMONIA       0.77      0.98      0.86       390

    accuracy                           0.80       624
   macro avg       0.86      0.74      0.76       624
weighted avg       0.84      0.80      0.79       624

AUROC: 0.9389 | AUPRC: 0.9595
Macro F1: 0.7611 | Weighted F1: 0.7865
Sensitivity: 0.9846 | Specificity: 0.5043 | Precision: 0.7680

=== EfficientNetB0 @ threshold 0.690 ===
[[184  50]
 [ 30 360]]
              precision    recall  f1-score   support

      NORMAL       0.86      0.79      0.82       234
   PNEUMONIA       0.88      0.92      0.90       390

    accuracy                           0.87       624
   macro avg       0.87      0.85      0.86       624
weighted avg       0.87      0.87      0.87       624

AUROC: 0.9389 | AUPRC: 0.9595
Macro F1: 0.8607 | Weighted F1: 0.8705
Sensitivity: 0.9

({'Model': 'EfficientNetB0',
  'Flip': None,
  'Class Weights': 'None',
  'Threshold': 0.25,
  'AUROC': 0.9388998465921543,
  'AUPRC': 0.9595459869264713,
  'Accuracy': 0.8044871794871795,
  'Macro F1': 0.7610696127047893,
  'Weighted F1': 0.7865325466072438,
  'Sensitivity': 0.9846153846153847,
  'Specificity': 0.5042735042735043,
  'Precision': 0.768,
  'TN': 118,
  'FP': 116,
  'FN': 6,
  'TP': 384,
  'Mode': 'Screening',
  'Params': 4050852},
 {'Model': 'EfficientNetB0',
  'Flip': None,
  'Class Weights': 'None',
  'Threshold': 0.69,
  'AUROC': 0.9388998465921543,
  'AUPRC': 0.9595459869264713,
  'Accuracy': 0.8717948717948718,
  'Macro F1': 0.8607142857142858,
  'Weighted F1': 0.8705357142857142,
  'Sensitivity': 0.9230769230769231,
  'Specificity': 0.7863247863247863,
  'Precision': 0.8780487804878049,
  'TN': 184,
  'FP': 50,
  'FN': 30,
  'TP': 360,
  'Mode': 'Diagnostic',
  'Params': 4050852})

EfficientNetB0 was evaluated as a lightweight transfer-learning architecture for pneumonia detection from chest X-ray images. Despite achieving strong validation performance during head-only training, fine-tuning the backbone did not yield additional gains, indicating that pretrained features were already well aligned with the task. At a screening-oriented operating point, EfficientNetB0 achieved high sensitivity (98.5%) but at the cost of a relatively high false-positive rate, while a more conservative threshold improved specificity but resulted in a substantial increase in missed pneumonia cases.

In [21]:
import pandas as pd
from pathlib import Path

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

pd.DataFrame([eff_screen_res, eff_diag_res]).to_csv(
    out_dir / "efficientnet_results.csv",
    index=False
)
